# 04 Data Augmentation Impact Analysis
This notebook compares a **Baseline Model** (no augmentation) against an **Augmented Model** (heavy transformations) to evaluate robustness and generalization.

In [1]:
import os
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
import mlflow
import bentoml
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
import numpy as np

from pneumonia_classifier.ml.model.arch import Net

## Configuration

In [2]:


# --- 1. Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = r'j:\Users\ayush\Desktop\code\pneumonia_classifier\artifacts\02_12_2025_08_52_04\data_ingestion\data\data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
INPUT_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS_PER_FOLD = 25 # Increased from 5
LR = 0.005 # Slightly lower to avoid overshooting

mlflow.set_experiment("augmentation_impact_analysis")

<Experiment: artifact_location='file:///j:/Users/ayush/Desktop/code/pneumonia_classifier/notebooks/mlruns/2', creation_time=1772278540841, experiment_id='2', last_update_time=1772278540841, lifecycle_stage='active', name='augmentation_impact_analysis', tags={}, workspace='default'>

## Define Transforms

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224), # Randomly shift the image
    transforms.RandomHorizontalFlip(p=0.5), # Left/Right flip
    transforms.RandomRotation(degrees=15), # Slight tilts
    transforms.ColorJitter(brightness=0.1, contrast=0.1), # Lighting changes
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


class AugmentedSubset(torch.utils.data.Dataset):
    """Wrapper to apply specific transforms to a Subset"""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)



## Training Loop

In [4]:

def run_experiment(name, transform_to_use):
    print(f"\nStarting Experiment: {name}")
    with mlflow.start_run(run_name=name):
        
        base_dataset = datasets.ImageFolder(TRAIN_DIR, transform=None) 
        targets = base_dataset.targets
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_f1_scores = []
        fold_acc_scores = []
        
        for fold, (train_ids, val_ids) in enumerate(skf.split(np.zeros(len(targets)), targets)):
            print(f"\n--- Fold {fold + 1}/5 ---")
            
            raw_train_sub = Subset(base_dataset, train_ids)
            raw_val_sub = Subset(base_dataset, val_ids)
            
            train_sub = AugmentedSubset(raw_train_sub, transform=transform_to_use)
            val_sub = AugmentedSubset(raw_val_sub, transform=val_transform)
            
            train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True,num_workers=0)
            val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False,num_workers=0)
            
            model = Net().to(DEVICE)
            optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)
            
            for epoch in range(1, EPOCHS_PER_FOLD + 1):
                model.train()
                for data, target in train_loader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    optimizer.zero_grad()
                    output = model(data)
                    loss = F.nll_loss(output, target)
                    loss.backward()
                    optimizer.step()
            
            # Validation Loop
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(DEVICE), target.to(DEVICE)
                    output = model(data)
                    pred = output.argmax(dim=1).cpu().numpy()
                    all_preds.extend(pred)
                    all_targets.extend(target.cpu().numpy())
            
            f1 = f1_score(all_targets, all_preds)
            acc = accuracy_score(all_targets, all_preds)
            fold_f1_scores.append(f1)
            fold_acc_scores.append(acc)
            
            mlflow.log_metric(f"fold_{fold+1}_f1", f1)
            mlflow.log_metric(f"fold_{fold+1}_acc", acc)
            print(f"Fold {fold + 1} - Acc: {100.*acc:.2f}%, F1: {f1:.4f}")
            
        mean_f1 = np.mean(fold_f1_scores)
        mean_acc = np.mean(fold_acc_scores)
        mlflow.log_metric("mean_cv_f1", mean_f1)
        mlflow.log_metric("mean_cv_acc", mean_acc)
        
        print(f"\nExperiment {name} Complete.")
        print(f"Mean CV Accuracy: {100.*mean_acc:.2f}%, Mean CV F1: {mean_f1:.4f}\n")
        
        mlflow.pytorch.log_model(model, "model")

        # --- NEW: Save the PyTorch model to BentoML ---
        bento_model = bentoml.pytorch.save_model(
            f"pneumonia_classifier_aug_{name}", # Differentiating the model name per experiment
            model,
            signatures={
                "__call__": {"batchable": True, "batch_dim": 0}
            }
        )
        print(f"Model saved to BentoML: {bento_model.tag}")
        mlflow.log_param("bento_model_tag", str(bento_model.tag))

## Run Comparisons

In [5]:
run_experiment("Baseline_No_Augmentation", val_transform)
run_experiment("Augmented_Heavy", train_transform)


Starting Experiment: Baseline_No_Augmentation



--- Fold 1/5 ---
Fold 1 - Acc: 92.86%, F1: 0.9268

--- Fold 2/5 ---
Fold 2 - Acc: 97.62%, F1: 0.9756

--- Fold 3/5 ---
Fold 3 - Acc: 92.86%, F1: 0.9333

--- Fold 4/5 ---
Fold 4 - Acc: 95.24%, F1: 0.9500

--- Fold 5/5 ---
Fold 5 - Acc: 97.62%, F1: 0.9767

Experiment Baseline_No_Augmentation Complete.
Mean CV Accuracy: 95.24%, Mean CV F1: 0.9525



2026/03/03 03:19:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 03:19:41 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
C:\Users\user\AppData\Local\Temp\ipykernel_7264\1926504284.py:68: BentoMLDeprecationWarning: `bentoml.pytorch` is deprecated since v1.4 and will be removed in a future version.
  bento_model = bentoml.pytorch.save_model(


Model saved to BentoML: pneumonia_classifier_aug_baseline_no_augmentation:zvyot3qwqgnsfahb

Starting Experiment: Augmented_Heavy

--- Fold 1/5 ---
Fold 1 - Acc: 95.24%, F1: 0.9524

--- Fold 2/5 ---
Fold 2 - Acc: 100.00%, F1: 1.0000

--- Fold 3/5 ---
Fold 3 - Acc: 80.95%, F1: 0.7647

--- Fold 4/5 ---
Fold 4 - Acc: 92.86%, F1: 0.9302

--- Fold 5/5 ---


2026/03/03 03:40:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Fold 5 - Acc: 92.86%, F1: 0.9231

Experiment Augmented_Heavy Complete.
Mean CV Accuracy: 92.38%, Mean CV F1: 0.9141



2026/03/03 03:40:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Model saved to BentoML: pneumonia_classifier_aug_augmented_heavy:wypxe3qwqsx75ahb


## Conclusion
Open the MLflow UI to compare the convergence speed and final F1-Score. Generally, augmentation improves robustness against real-world variations.